In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import pickle

MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


SET DATASET PATH

In [ ]:
data_dir = "/content/drive/MyDrive/food_dataset"

Load Dataset

In [ ]:
img_size = 224
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2147 images belonging to 36 classes.
Found 524 images belonging to 36 classes.


Build Model

In [ ]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Train Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

Epoch 1/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.5612 - loss: 1.7299 - val_accuracy: 0.8454 - val_loss: 0.5781
Epoch 2/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.8472 - loss: 0.5264 - val_accuracy: 0.8550 - val_loss: 0.4279
Epoch 3/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.9031 - loss: 0.3343 - val_accuracy: 0.8779 - val_loss: 0.3710
Epoch 4/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.9148 - loss: 0.2680 - val_accuracy: 0.8798 - val_loss: 0.3333
Epoch 5/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.9367 - loss: 0.2005 - val_accuracy: 0.8607 - val_loss: 0.3919
Epoch 6/15
68/68 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.9502 - loss: 0.1593 - val_accuracy: 0.8874 - val_loss: 0.3384


Save Model

In [ ]:
pickle.dump(train_data.class_indices, open("/content/drive/MyDrive/class_indices.pkl", "wb"))

In [ ]:
model.save("/content/drive/MyDrive/food_model.h5")